In [1]:
path_pattern = "/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/source_data/initial_dataset/*.parquet"

In [2]:
import polars as pl
import glob

all_files = sorted(glob.glob(path_pattern))
first_x = all_files[3:10]
if first_x:
    df_deps = pl.read_parquet(first_x, n_rows=100)
    print(f"Loaded {len(first_x)} files.")

Loaded 7 files.


In [3]:
from requests import Session
from datetime import datetime

def get_metrics(row_dict):
    dep_pkg = row_dict["dep_name"]
    dep_ver = row_dict["dep_version"]
    snapshot_date = row_dict["SnapshotAt"]
    session = Session()
    result = {"ttu_days": None, "ttr_days": None}

# Initialize defaults
    result = {"ttu_days": None, "ttr_days": None}

    try:
        # PyPI
        res = session.get(f"https://pypi.org/pypi/{dep_pkg}/json", timeout=10)
        if res.status_code != 200:
            return result
            
        data = res.json()
        releases = data.get('releases', {})
        latest_ver = data['info']['version']
        
        # TTU
        if latest_ver in releases and releases[latest_ver]:
            up_time = releases[latest_ver][0]['upload_time'].replace("Z", "")
            latest_release_date = datetime.fromisoformat(up_time)
            result["ttu_days"] = (latest_release_date - snapshot_date).days

        # 2. TTR
        osv_query = {"version": dep_ver, "package": {"name": dep_pkg, "ecosystem": "PyPI"}}
        osv_res = session.post("https://api.osv.dev/v1/query", json=osv_query, timeout=10).json()
        
        if 'vulns' in osv_res:
            vuln = osv_res['vulns'][0]
            pub_date = datetime.fromisoformat(vuln['published'].replace("Z", ""))
            
            for affected in vuln.get('affected', []):
                for r in affected.get('ranges', []):
                    for event in r.get('events', []):
                        fixed_ver = event.get('fixed')
                        if fixed_ver in releases and releases[fixed_ver]:
                            fix_time = releases[fixed_ver][0]['upload_time'].replace("Z", "")
                            fix_date = datetime.fromisoformat(fix_time)
                            result["ttr_days"] = max(0, (fix_date - pub_date).days)
                            return result
                            
        return result
    except Exception as e:
        print(f"Error occurred while processing {dep_pkg}: {e}")
        return result



In [4]:
from concurrent.futures import ThreadPoolExecutor
rows_to_process = df_deps.select(["dep_name", "dep_version", "SnapshotAt"]).to_dicts()
with ThreadPoolExecutor(max_workers=15) as executor:
    metrics_list = list(executor.map(get_metrics, rows_to_process))
df_metrics = pl.DataFrame(metrics_list)
df_results = pl.concat([df_deps, df_metrics], how="horizontal")

In [5]:
df_results.head(30)

SnapshotAt,package_name,package_version,package_published_at,dep_name,dep_version,MinimumDepth,github_repo,ttu_days,ttr_days
datetime[μs],str,str,datetime[μs],str,str,i64,str,i64,i64
2023-06-05 21:01:37.724475,"""pymiscutils""","""0.3.14""",null,"""msoffcrypto-tool""","""5.0.1""",2,"""matthewgdv/miscutils""",951,null
2023-06-05 21:01:37.724475,"""pymiscutils""","""0.3.14""",null,"""numpy""","""1.24.3""",1,"""matthewgdv/miscutils""",1007,null
2023-06-05 21:01:37.724475,"""pymiscutils""","""0.3.14""",null,"""olefile""","""0.46.0""",3,"""matthewgdv/miscutils""",178,null
2023-06-05 21:01:37.724475,"""pymiscutils""","""0.3.14""",null,"""pandas""","""2.0.2""",2,"""matthewgdv/miscutils""",988,null
2023-06-05 21:01:37.724475,"""pymiscutils""","""0.3.14""",null,"""parsedatetime""","""2.6.0""",2,"""matthewgdv/miscutils""",-1100,null
…,…,…,…,…,…,…,…,…,…
2023-06-05 21:01:37.724475,"""pymiscutils""","""0.3.14""",null,"""typing-extensions""","""4.6.3""",3,"""matthewgdv/miscutils""",811,null
2023-06-05 21:01:37.724475,"""pymiscutils""","""0.3.14""",null,"""tzdata""","""2023.3.0""",3,"""matthewgdv/miscutils""",921,null
2023-06-05 21:01:37.724475,"""pymiscutils""","""0.3.14""",null,"""urllib3""","""2.0.2""",2,"""matthewgdv/miscutils""",946,0


In [6]:
mttr_mttu_df = df_results.group_by("package_name").agg([
    pl.col("ttu_days").mean().alias("mttu"),
    pl.col("ttr_days").mean().alias("mttr"),
    pl.len().alias("dep_count"),
    pl.col("ttr_days").drop_nulls().count().alias("vulnerable_dep_count")
])

mttr_mttu_df.head(30)


package_name,mttu,mttr,dep_count,vulnerable_dep_count
str,f64,f64,u32,u32
"""pymmaster""",714.770833,0.0,48,9
"""pymkv""",-2171.0,null,2,0
"""pymm-eventserver""",727.411765,0.0,17,1
"""pymiscutils""",479.517241,0.0,29,5
"""pymlconf""",843.0,null,4,0


In [7]:
output_dir = "./data/source_data/ttu_ttr/"

df_results.write_parquet(
    output_dir,
    partition_by="package_name"
)

In [8]:
path_pattern = "/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/source_data/pypi_scorecards/*.parquet"
all_files = sorted(glob.glob(path_pattern))
first_x = all_files[:10]
if first_x:
    df_scorecard = pl.read_parquet(first_x)
    print(f"Loaded {len(first_x)} files.")

Loaded 10 files.


In [9]:
df_scorecard.head(30)

scorecard_date,repo_name,repo_commit,scorecard_version,aggregate_score,check_name,check_score
date,str,str,str,f64,str,i64
2023-10-30,"""github.com/keybase/pykeybasebo…","""38c84aac41fc03d4d2693fb316288d…","""v4.13.1-26-g478f347e""",2.5,"""Code-Review""",6
2023-10-30,"""github.com/keybase/pykeybasebo…","""38c84aac41fc03d4d2693fb316288d…","""v4.13.1-26-g478f347e""",2.5,"""Pinned-Dependencies""",-1
2023-10-30,"""github.com/paweltroka/signalr-…","""effa7347016174e460d3c3e3f2a9d0…","""v4.13.1-26-g478f347e""",2.5,"""Binary-Artifacts""",10
2023-10-30,"""github.com/kaixuyang/lunarcale…","""888497987ffe7fecca94f544c03e56…","""v4.13.1-26-g478f347e""",3.0,"""Dangerous-Workflow""",-1
2023-10-30,"""github.com/jjmontesl/codenamiz…","""5af5f8bca3b39d18806a051fd112ea…","""v4.13.1-26-g478f347e""",3.0,"""CII-Best-Practices""",0
…,…,…,…,…,…,…
2023-10-30,"""github.com/umccr/libumccr""","""6434ff2a7a6b2156bfbf619a9efea6…","""v4.13.1-26-g478f347e""",4.8,"""Fuzzing""",0
2023-10-30,"""github.com/modeci/modelspec""","""e1587065ddfc33a24157ad82396c3b…","""v4.13.1-26-g478f347e""",5.3,"""Fuzzing""",0
2023-10-30,"""github.com/sphinx-doc/sphinx-i…","""d572656e871d3f8850f8a414cdc2e8…","""v4.13.1-26-g478f347e""",5.3,"""Branch-Protection""",-1


In [10]:
df_deps = df_deps.with_columns(
    pl.col("github_repo").str.replace("https://", "").str.replace("github.com/", ""),
    pl.col("package_published_at").cast(pl.Datetime)
)

dtype = df_scorecard.schema.get("scorecard_date")
if dtype is not None and "Date" in str(dtype):
    df_scorecard = df_scorecard.with_columns(
        pl.col("repo_name").str.replace("https://", "").str.replace("github.com/", ""),
        pl.col("scorecard_date").cast(pl.Datetime)
    )
else:
    df_scorecard = df_scorecard.with_columns(
        pl.col("repo_name").str.replace("https://", "").str.replace("github.com/", ""),
        pl.col("scorecard_date").str.to_datetime().cast(pl.Datetime)
    )


df_deps = df_deps.sort("package_published_at")
df_scorecard = df_scorecard.sort("scorecard_date")


df_joined = df_deps.join_asof(
    df_scorecard,
    left_on="package_published_at",
    right_on="scorecard_date",
    by_left="github_repo",
    by_right="repo_name",
    strategy="backward"
)

/tmp/ipykernel_64802/3577651225.py:23: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  df_joined = df_deps.join_asof(


In [11]:
# df_scorecard = df_scorecard.with_columns(
#     pl.col("repo_name").str.replace(r"^github\.com/", "").alias("match_repo")
# )
# df_deps = df_deps.with_columns(
#     pl.col("github_repo").alias("match_repo")
# )

# df_scorecard_collapsed = (
#     df_scorecard
#     .group_by("match_repo")
#     .agg([
#         pl.col("aggregate_score").first(),  # Keep the main score
#         # Optional: Get specific scores as columns
#         pl.col("check_score").filter(pl.col("check_name") == "Code-Review").first().alias("score_code_review"),
#         pl.col("check_score").filter(pl.col("check_name") == "Maintained").first().alias("score_maintained")
#     ])
# )

# joined_df = df_deps.join(df_scorecard_collapsed, on="match_repo", how="left")